# Ingestion + Chunking Pipeline

This notebook covers Steps 3–4 of the RAID roadmap:

1. Load ingested article/point records from `data/processed/*.jsonl`
2. Run **two** chunking strategies (recursive baseline + article-aware)
3. Persist `data/chunks_*.jsonl` and inspect counts / length stats


In [ ]:
from glob import glob
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from app.chunking import (
    chunk_by_article,
    chunk_recursive,
    load_processed_records,
    summarize_chunks,
)

processed_paths = sorted(glob(str(ROOT / "data" / "processed" / "*.jsonl")))
records = load_processed_records(processed_paths)
print(f"Loaded {len(records)} records from {len(processed_paths)} files")
for path in processed_paths:
    print(" -", Path(path).name)


## Step 4 — Two chunking strategies

| Strategy | Method | Output |
|----------|--------|--------|
| Baseline | `RecursiveCharacterTextSplitter` on full docs | `chunks_recursive.jsonl` |
| Primary | One chunk per `Статья`/`Пункт`, sub-split if >1500 chars | `chunks_article.jsonl` → `chunks.jsonl` |


In [ ]:
import json

out_dir = ROOT / "data"
out_dir.mkdir(parents=True, exist_ok=True)

recursive_chunks = chunk_recursive(records, chunk_size=800, chunk_overlap=100)
article_chunks = chunk_by_article(records, max_chars=1500, chunk_overlap=100)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

write_jsonl(out_dir / "chunks_recursive.jsonl", recursive_chunks)
write_jsonl(out_dir / "chunks_article.jsonl", article_chunks)
write_jsonl(out_dir / "chunks.jsonl", article_chunks)  # default for indexing

print("recursive:", summarize_chunks(recursive_chunks))
print("article:  ", summarize_chunks(article_chunks))
print("sample article chunk_id:", article_chunks[1]["chunk_id"])
print("sample section_title:", article_chunks[1]["section_title"])
print("sample text preview:", article_chunks[1]["text"][:240], "...")
